# Day 2 · §2.8.5 — How CNNs Work  *(Instructor Demo)*

**AI for Cybersecurity Professionals · Day 2: AI for Defense**

> **Instructor-led demo, not a graded student lab.** Runs on Google Colab. We use the small
> built-in **`load_digits`** image set (8×8 handwritten digits) because the usual MNIST download
> is often broken on Colab. The heavier, full-resolution version runs on the instructor's
> Mac Mini M4.

### Why images need something new
- A normal ("fully-connected") network treats every pixel as an unrelated number, so it **ignores
  spatial structure** — that a pixel's *neighbors* matter. It also doesn't scale: a 1000×1000
  image = a million inputs.
- A **Convolutional Neural Network (CNN)** fixes this with two ideas:
  - **Convolution / filters:** a tiny window (say 3×3) **slides** across the image looking for a
    local pattern — an edge, then a corner, then a shape, then a whole object as layers stack.
  - **Pooling:** shrink the image while keeping the strongest signals, so the network focuses on
    *what* is present, not exactly *where*.
- **Security relevance:** malware rendered **as an image** and classified by a CNN; deepfake and
  image-forensics detection (which sets up the Day-3 deepfake demo, §3.7).


## Step 1 — Tools

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
print("TensorFlow version:", tf.__version__)

## Step 2 — Load the tiny digit images and look at a few

In [ ]:
digits = load_digits()
print("images:", digits.images.shape, " (samples, 8, 8) — each is an 8x8 grid of pixels")
print("labels:", digits.target.shape, " values 0-9")

fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for ax, img, lab in zip(axes, digits.images, digits.target):
    ax.imshow(img, cmap="gray"); ax.set_title(str(lab)); ax.axis("off")
plt.suptitle("What the CNN sees (8x8 handwritten digits)"); plt.show()

## Step 3 — Prepare the images

Three quick prep steps: (1) reshape each image to add a "channel" dimension the CNN expects,
(2) scale pixel values to 0-1 (Lab 3b again — nets need scaled inputs), (3) one-hot encode the
labels (turn "7" into a 10-slot vector that is 1 in position 7).


In [ ]:
X = digits.images.reshape(-1, 8, 8, 1).astype("float32") / 16.0   # 16 = max pixel value here
y = to_categorical(digits.target, num_classes=10)                 # 0-9 -> 10-slot vectors

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)
print("Train:", X_tr.shape, " Test:", X_te.shape)

## Step 4 — Build a small CNN

`Conv2D` = the sliding-filter layer; `MaxPooling2D` = the shrink step; `Flatten` turns the 2-D
maps into a list; `Dense` layers make the final 10-way decision.


In [ ]:
model = Sequential([
    Conv2D(16, (3, 3), activation="relu", input_shape=(8, 8, 1)),  # 16 sliding 3x3 filters
    MaxPooling2D((2, 2)),                                           # shrink, keep strongest signals
    Flatten(),                                                      # 2-D maps -> one long list
    Dense(32, activation="relu"),
    Dense(10, activation="softmax"),                               # 10 outputs = digits 0-9
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

## Step 5 — Train and watch accuracy climb

In [ ]:
history = model.fit(X_tr, y_tr, validation_data=(X_te, y_te),
                    epochs=15, batch_size=32, verbose=0)
h = history.history
plt.figure(figsize=(7,4))
plt.plot(h["accuracy"], label="training", color="#1E5199")
plt.plot(h["val_accuracy"], label="validation", color="#00838F")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend()
plt.title("CNN learning to read digits"); plt.tight_layout(); plt.show()
print("Final validation accuracy: {:.2f}".format(h["val_accuracy"][-1]))

## Step 6 — Test it on a few unseen images

In [ ]:
proba = model.predict(X_te[:8], verbose=0)      # probabilities for 8 test images
pred  = proba.argmax(axis=1)                    # pick the most likely digit
true  = y_te[:8].argmax(axis=1)

fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for ax, img, p, t in zip(axes, X_te[:8].reshape(-1,8,8), pred, true):
    ax.imshow(img, cmap="gray"); ax.axis("off")
    ax.set_title(f"pred {p}\ntrue {t}", color=("green" if p==t else "red"), fontsize=9)
plt.suptitle("CNN predictions on unseen digits"); plt.show()

## Wrap-up
- A CNN reads images with **sliding filters** (find local patterns) + **pooling** (shrink while
  keeping what matters) — instead of treating pixels as unrelated numbers.
- Filters build up: edges → shapes → objects, layer by layer.
- **Security relevance:** malware-as-image classification and deepfake/image forensics — the
  jumping-off point for the Day-3 deepfake demo (§3.7).
- On this 8×8 toy set a CNN already reads digits well; the same architecture, scaled up on the
  **M4**, handles real images.
